In [ ]:
# CELDA 1 — Verificar GPU disponible
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU encontrada — asegúrate de ir a Runtime > Change runtime type > GPU')

In [ ]:
# CELDA 2 — Instalar Unsloth
# Esto tarda ~2-3 minutos la primera vez
!pip install unsloth --quiet
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo --quiet
print('Unsloth instalado correctamente')

In [ ]:
# CELDA 3 — Montar Google Drive y usar carpeta existente
from google.colab import drive
from googleapiclient.discovery import build
from google.colab import auth
import os

drive.mount('/content/drive')
auth.authenticate_user()

# Obtener nombre de la carpeta por ID (asume que está directamente bajo MyDrive)
FOLDER_ID = '1HNgLVG6b907WOtDxSgP56gcjylsDd0tq'
service = build('drive', 'v3', cache_discovery=False)

meta = service.files().get(fileId=FOLDER_ID, fields='name').execute()
BASE_DIR       = f'/content/drive/MyDrive/{meta["name"]}'
CHECKPOINT_DIR = f'{BASE_DIR}/qwen3-0.6b/checkpoints'
ADAPTER_DIR    = f'{BASE_DIR}/qwen3-0.6b/adapter'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR,    exist_ok=True)

print(f'✅ Google Drive montado')
print(f'📁 Base:         {BASE_DIR}')
print(f'📁 Checkpoints:  {CHECKPOINT_DIR}')
print(f'📁 Adapter:      {ADAPTER_DIR}')

In [ ]:
# CELDA 4 — Clonar repositorio CARpsy (contiene el dataset)
import os

REPO_URL = 'https://github.com/gazzimon/CARpsy.git'
REPO_DIR = '/content/CARpsy'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}

# Verificar que el dataset está presente
train_path = f'{REPO_DIR}/data/splits/train.jsonl'
assert os.path.exists(train_path), f'❌ No se encontró: {train_path}'
print(f'✅ Repo listo en: {REPO_DIR}')
print(f'✅ Dataset encontrado: {train_path}')

In [ ]:
# CELDA 5 — Cargar modelo Qwen3-0.6B con Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-0.6B',
    max_seq_length = 512,
    dtype = None,           # Auto-detect: float16 en T4, bfloat16 en A100
    load_in_4bit = True,    # 4-bit para caber en T4 (16GB VRAM)
)
print(f'Modelo cargado. VRAM usada: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
# CELDA 6 — Configurar LoRA (mismos parámetros que en lora_config.yaml)
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,                          # LoRA rank
    lora_alpha = 16,                 # LoRA alpha
    lora_dropout = 0.05,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention (equivalente a attn_q/k/v/o)
    ],
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',  # Reduce VRAM en 30%
    random_state = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}% del total)')

In [ ]:
# CELDA 7 — Preparar dataset ChatML
import json, os, subprocess
from datasets import Dataset

TRAIN_PATH = '/content/train.jsonl'
GITHUB_RAW  = 'https://raw.githubusercontent.com/gazzimon/CARpsy/main/data/splits/train.jsonl'

# Descarga desde GitHub si no existe localmente
if not os.path.exists(TRAIN_PATH):
    print('Descargando train.jsonl desde GitHub...')
    subprocess.run(['wget', '-q', GITHUB_RAW, '-O', TRAIN_PATH], check=True)
    print(f'✅ Descargado en {TRAIN_PATH}')
else:
    print(f'✅ Usando archivo existente: {TRAIN_PATH}')

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def format_example(example):
    return tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )

raw = load_jsonl(TRAIN_PATH)
formatted = [{'text': format_example(ex)} for ex in raw]
dataset = Dataset.from_list(formatted)

print(f'Dataset preparado: {len(dataset)} ejemplos')
print('\nEjemplo formateado:')
print(dataset[0]['text'][:400])

In [ ]:
# CELDA 8 — Entrenar
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import glob

# Detectar checkpoint más reciente (sort numérico, no alfabético)
checkpoints = glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*')
checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f'▶ Resumiendo desde: {resume_from}')
else:
    print('▶ Entrenamiento desde cero')

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = 'text',
    max_seq_length     = 512,
    dataset_num_proc   = 2,
    args = SFTConfig(
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,
        num_train_epochs             = 3,
        learning_rate                = 2e-4,
        weight_decay                 = 1e-2,
        lr_scheduler_type            = 'cosine',
        warmup_steps                 = 50,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 50,
        save_steps                   = 100,
        save_total_limit             = 3,
        output_dir                   = CHECKPOINT_DIR,
        optim                        = 'adamw_8bit',
        seed                         = 42,
        report_to                    = 'none',
        max_seq_length               = 512,
    ),
)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# CELDA 9 — Guardar adapter en Google Drive
adapter_path = f'{ADAPTER_DIR}/carpsy-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adapter guardado en: {adapter_path}')

# CELDA 10 — Exportar a GGUF Q4_K_M (compatible con llama.cpp / QVAC Fabric)
import os

gguf_path = f'{ADAPTER_DIR}/carpsy-adapter'
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method='q4_k_m')

# Unsloth guarda los GGUFs en subcarpeta _gguf
gguf_dir = f'{gguf_path}_gguf'
if not os.path.exists(gguf_dir):
    gguf_dir = ADAPTER_DIR

gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')]
print('✅ Archivos GGUF generados:')
for f in gguf_files:
    size = os.path.getsize(f'{gguf_dir}/{f}') / 1024**2
    print(f'  {gguf_dir}/{f} — {size:.1f} MB')
print('\n🎉 Fine-tuning completo. Descarga el .gguf desde Google Drive.')